# ParkInsight — Preprocessing Dataset Transaksi Parkir Triguna

Notebook ini mengubah `data_trans_6_bulan_terakhir.xlsx` menjadi dataset siap latih untuk model **Random Forest Regression** (`rf_motor` & `rf_car`).

**Alur:**
1. Mount Google Drive & konfigurasi path
2. Baca Excel (streaming XML, cepat untuk ±482 ribu baris)
3. Cleaning: duplikat, timestamp tidak valid, durasi `<= 0` / `> 24 jam`, `BOX`
4. Buang anomali `Pos Masuk = PK*` (durasi sintetis tepat 60 menit)
5. Pisahkan Motor (semua `PM*`) & Mobil (`PM*` kecuali `PM1D MBL`)
6. Agregasi interval 5 menit kontinu → tabel `occupancy` (clip `[0, C]` + flag anomali)
7. Fitur waktu, lag, rolling 15 menit, dan target `t+10`, `t+15`, `t+30`
8. Split kronologis 80:20 & simpan CSV ke Google Drive

## 1. Mount Google Drive

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Running di Google Colab' if IN_COLAB else 'Running di lokal')

## 2. Import Library & Konfigurasi

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)

# ---------------- PATH ----------------
COLAB_PROJECT_DIR = Path('/content/drive/MyDrive/Coolyeah Daffa/Topik Khusus/Project Capstone')
LOCAL_PROJECT_DIR = Path(r'd:\Folder Dappaa\Coolyeah\Topik Khusus\Project Capstone')  # fallback lokal

PROJECT_DIR = COLAB_PROJECT_DIR if IN_COLAB else LOCAL_PROJECT_DIR
INPUT_XLSX = PROJECT_DIR / 'dataset' / 'data_trans_6_bulan_terakhir.xlsx'
OUTPUT_DIR = PROJECT_DIR / 'ml' / 'output'

# ---------------- PARAMETER ----------------
MOTOR_CAPACITY = 3000
CAR_CAPACITY = 50

EXCLUDE_PK_AS_ENTRY = True          # wajib: Pos Masuk PK* = anomali durasi sintetis 60 menit
EXCLUDE_PM_AS_EXIT = False          # opsional: ±924 baris keluar lewat PM*
EXCLUDE_REKTOR_FOR_MOTOR = True     # opsional: buang PM1E REKTOR untuk motor
EXCLUDED_CAR_ENTRY_GATES = ['PM1D MBL']  # anomali skala mobil (puncak 165-348 mobil)

CAR_CLIP_TO_CAPACITY = True         # O_t = min(C, max(0, ...)) sesuai Persamaan (1)
DROP_CAR_OVERFLOW_DAYS = False      # True = buang hari dengan mobil > 120% kapasitas

FREQ = '5min'
OPS_START_HOUR = 6                  # 06:00 WIB
OPS_END_HOUR = 22                   # s/d 21:55 WIB
DROP_FIRST_DAY_WARMUP = True
TRAIN_RATIO = 0.80

print('Input  :', INPUT_XLSX)
print('Output :', OUTPUT_DIR)
assert INPUT_XLSX.exists(), f'File tidak ditemukan: {INPUT_XLSX}'

## 3. Baca Dataset Excel (Streaming XML)

In [ ]:
def load_excel_transactions_fast(xlsx_path):
    ns = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
    with zipfile.ZipFile(xlsx_path) as zf:
        ss_root = ET.fromstring(zf.read('xl/sharedStrings.xml'))
        shared = [''.join(t.text or '' for t in si.iter() if t.tag.endswith('t')) for si in ss_root]
        rows = []
        with zf.open('xl/worksheets/sheet1.xml') as f:
            for _, elem in ET.iterparse(f, events=('end',)):
                if elem.tag != f'{ns}row':
                    continue
                if int(elem.attrib.get('r', 0)) > 1:   # lewati header
                    rec = {}
                    for cell in elem:
                        col = ''.join(ch for ch in cell.attrib.get('r', '') if ch.isalpha())
                        if col in ('A', 'B', 'C', 'D', 'E'):
                            v = cell.find(f'{ns}v')
                            val = v.text if v is not None else None
                            if cell.attrib.get('t', '') == 's' and val is not None:
                                val = shared[int(val)]
                            rec[col] = val
                    rows.append((rec.get('A'), rec.get('B'), rec.get('C'), rec.get('D'), rec.get('E')))
                elem.clear()

    df = pd.DataFrame(rows, columns=['waktu_masuk_raw', 'pos_masuk', 'waktu_keluar_raw', 'pos_keluar', 'kendaraan'])
    for raw, col in (('waktu_masuk_raw', 'waktu_masuk'), ('waktu_keluar_raw', 'waktu_keluar')):
        df[col] = pd.to_datetime(pd.to_numeric(df[raw], errors='coerce'), unit='D', origin='1899-12-30')
    return df[['waktu_masuk', 'pos_masuk', 'waktu_keluar', 'pos_keluar', 'kendaraan']]

df_raw = load_excel_transactions_fast(INPUT_XLSX)
print(f'Total baris mentah: {len(df_raw):,}')
df_raw.head()

## 4. Cleaning & Penghapusan Anomali

In [ ]:
audit = {'01_raw': len(df_raw)}

df = df_raw.drop_duplicates().copy()
audit['02_dedup'] = len(df)

valid_ts = (df['waktu_masuk'].notna() & df['waktu_keluar'].notna()
            & (df['waktu_masuk'].dt.year >= 2020) & (df['waktu_keluar'].dt.year >= 2020))
df = df[valid_ts].copy()
audit['03_valid_timestamp'] = len(df)

df['durasi_detik'] = (df['waktu_keluar'] - df['waktu_masuk']).dt.total_seconds()
df = df[(df['durasi_detik'] > 0) & (df['durasi_detik'] <= 86400)].copy()
audit['04_durasi_0_24jam'] = len(df)

for c in ('kendaraan', 'pos_masuk', 'pos_keluar'):
    df[c] = df[c].astype(str).str.strip()
df = df[df['kendaraan'] != 'BOX'].copy()
audit['05_drop_box'] = len(df)

if EXCLUDE_PK_AS_ENTRY:
    df = df[df['pos_masuk'].str.startswith('PM')].copy()
    audit['06_drop_pos_masuk_PK'] = len(df)

if EXCLUDE_PM_AS_EXIT:
    df = df[df['pos_keluar'].str.startswith('PK')].copy()
    audit['07_drop_pos_keluar_PM'] = len(df)

# Jenis kendaraan ditentukan dari kolom Kendaraan, BUKAN dari nama pos
df['vehicle_type'] = np.where(df['kendaraan'].str.contains('MOTOR', case=False, na=False), 'motor', 'car')
df_clean = df

for k, v in audit.items():
    print(f'{k:25s}: {v:,}')

## 5. Pisahkan Dataset Motor & Mobil

In [ ]:
motor_mask = df_clean['vehicle_type'] == 'motor'
if EXCLUDE_REKTOR_FOR_MOTOR:
    motor_mask &= df_clean['pos_masuk'] != 'PM1E REKTOR'
df_motor = df_clean[motor_mask].copy()

car_mask = (df_clean['vehicle_type'] == 'car') & (~df_clean['pos_masuk'].isin(EXCLUDED_CAR_ENTRY_GATES))
df_car = df_clean[car_mask].copy()

print(f'Motor: {len(df_motor):,} transaksi')
print(df_motor['pos_masuk'].value_counts().to_string(), '\n')
print(f'Mobil: {len(df_car):,} transaksi')
print(df_car['pos_masuk'].value_counts().to_string())

## 6. Agregasi 5 Menit & Tabel `occupancy`

In [ ]:
def aggregate_flow(df_v, grid, prefix, capacity):
    m_in = df_v['waktu_masuk'].dt.floor(FREQ).value_counts().reindex(grid, fill_value=0)
    m_out = df_v['waktu_keluar'].dt.floor(FREQ).value_counts().reindex(grid, fill_value=0)
    net = m_in - m_out
    raw_occ = net.cumsum().astype(int)          # okupansi kontinu lintas hari
    return pd.DataFrame({
        f'{prefix}_in': m_in.astype(int),
        f'{prefix}_out': m_out.astype(int),
        f'{prefix}_net_flow': net.astype(int),
        f'{prefix}_occupancy_raw': raw_occ,
        f'{prefix}_occupancy': raw_occ.clip(0, capacity).astype(int),
        f'{prefix}_is_anomaly': (raw_occ < 0) | (raw_occ > capacity),
    }, index=grid)

start = df_motor['waktu_masuk'].dt.normalize().min()
end = df_motor['waktu_masuk'].dt.normalize().max() + pd.Timedelta(hours=23, minutes=55)
grid = pd.date_range(start, end, freq=FREQ)

occ = pd.concat([
    aggregate_flow(df_motor, grid, 'motor', MOTOR_CAPACITY),
    aggregate_flow(df_car, grid, 'car', CAR_CAPACITY),
], axis=1)
occ.index.name = 'interval_start'
occ = occ.reset_index()

if not CAR_CLIP_TO_CAPACITY:
    occ['car_occupancy'] = occ['car_occupancy_raw']

occ['total_occupancy'] = occ['motor_occupancy'] + occ['car_occupancy']
occ['is_anomaly'] = occ['motor_is_anomaly'] | occ['car_is_anomaly']
occ['anomaly_note'] = np.select(
    [occ['car_occupancy_raw'] > CAR_CAPACITY, occ['motor_occupancy_raw'] > MOTOR_CAPACITY],
    ['car_occupancy_exceeds_capacity', 'motor_occupancy_exceeds_capacity'],
    default='',
)

print(f'Total interval 5 menit : {len(occ):,}')
print(f'Max motor_occupancy    : {occ.motor_occupancy.max()} / {MOTOR_CAPACITY}')
print(f'Max car_occupancy_raw  : {occ.car_occupancy_raw.max()} (clipped {occ.car_occupancy.max()} / {CAR_CAPACITY})')
print(f'Interval anomali mobil : {(occ.car_occupancy_raw > CAR_CAPACITY).sum():,}')
occ.head()

## 7. Feature Engineering & Target (`t+10`, `t+15`, `t+30`)

In [ ]:
feat = occ.copy()
feat['hour'] = feat['interval_start'].dt.hour
feat['minute'] = feat['interval_start'].dt.minute
feat['minute_of_day'] = feat['hour'] * 60 + feat['minute']
feat['day_of_week'] = feat['interval_start'].dt.dayofweek   # 0=Senin .. 6=Minggu
feat['is_weekday'] = (feat['day_of_week'] < 5).astype(int)

for lag in (1, 2, 3):
    for col in ('motor_occupancy', 'car_occupancy', 'motor_in', 'car_in'):
        feat[f'{col}_lag_{lag}'] = feat[col].shift(lag)

for col in ('motor_in', 'motor_out', 'car_in', 'car_out'):
    feat[f'{col}_roll15'] = feat[col].rolling(3).sum()

for horizon, steps in ((10, 2), (15, 3), (30, 6)):
    feat[f'motor_occupancy_t+{horizon}'] = feat['motor_occupancy'].shift(-steps)
    feat[f'car_occupancy_t+{horizon}'] = feat['car_occupancy'].shift(-steps)

daily_car_max = feat.groupby(feat['interval_start'].dt.date)['car_occupancy_raw'].transform('max')
feat['is_car_overflow_day'] = daily_car_max > CAR_CAPACITY * 1.2

# Filter setelah lag/target dibentuk (agar tidak putus di batas jam operasional)
if DROP_FIRST_DAY_WARMUP:
    feat = feat[feat['interval_start'].dt.date > feat['interval_start'].dt.date.min()]
feat = feat[(feat['hour'] >= OPS_START_HOUR) & (feat['hour'] < OPS_END_HOUR)].copy()
if DROP_CAR_OVERFLOW_DAYS:
    feat = feat[~feat['is_car_overflow_day']].copy()

feat = feat.dropna(subset=['motor_occupancy_lag_3', 'car_occupancy_lag_3',
                           'motor_occupancy_t+30', 'car_occupancy_t+30']).reset_index(drop=True)
int_cols = [c for c in feat.columns if 'lag_' in c or 'roll15' in c or '_t+' in c]
feat[int_cols] = feat[int_cols].astype(int)

print(f'Dataset fitur: {feat.shape[0]:,} baris x {feat.shape[1]} kolom')
feat.head()

## 8. Definisi Fitur (`X`) & Target (`y`)

In [ ]:
FEATURE_COLS = [
    'hour', 'minute', 'minute_of_day', 'day_of_week', 'is_weekday',
    'motor_in', 'motor_out', 'motor_net_flow',
    'car_in', 'car_out', 'car_net_flow',
    'motor_occupancy', 'car_occupancy', 'total_occupancy',
    'motor_occupancy_lag_1', 'motor_occupancy_lag_2', 'motor_occupancy_lag_3',
    'car_occupancy_lag_1', 'car_occupancy_lag_2', 'car_occupancy_lag_3',
    'motor_in_lag_1', 'motor_in_lag_2', 'motor_in_lag_3',
    'car_in_lag_1', 'car_in_lag_2', 'car_in_lag_3',
    'motor_in_roll15', 'motor_out_roll15', 'car_in_roll15', 'car_out_roll15',
]
TARGET_COLS = {
    'motor': ['motor_occupancy_t+10', 'motor_occupancy_t+15', 'motor_occupancy_t+30'],
    'car':   ['car_occupancy_t+10', 'car_occupancy_t+15', 'car_occupancy_t+30'],
}
missing = [c for c in FEATURE_COLS + sum(TARGET_COLS.values(), []) if c not in feat.columns]
assert not missing, missing
print('Null pada fitur/target:', int(feat[FEATURE_COLS + sum(TARGET_COLS.values(), [])].isna().sum().sum()))

## 9. Split Kronologis 80:20 & Simpan CSV

In [ ]:
split_idx = int(len(feat) * TRAIN_RATIO)
train_df = feat.iloc[:split_idx].copy()
test_df = feat.iloc[split_idx:].copy()

print(f"Train: {len(train_df):,} baris ({train_df.interval_start.min()} s/d {train_df.interval_start.max()})")
print(f"Test : {len(test_df):,} baris ({test_df.interval_start.min()} s/d {test_df.interval_start.max()})")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
occ.to_csv(OUTPUT_DIR / 'occupancy_5min.csv', index=False)
feat.to_csv(OUTPUT_DIR / 'features_ml_ready.csv', index=False)
train_df.to_csv(OUTPUT_DIR / 'train_features.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test_features.csv', index=False)
print('Tersimpan di:', OUTPUT_DIR)

## 10. Visualisasi Cek Cepat (Opsional)

In [ ]:
import matplotlib.pyplot as plt

profile = feat[feat['is_weekday'] == 1].groupby('minute_of_day')[['motor_occupancy', 'car_occupancy']].mean()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col, cap, color in ((axes[0], 'motor_occupancy', MOTOR_CAPACITY, 'green'),
                            (axes[1], 'car_occupancy', CAR_CAPACITY, 'blue')):
    ax.plot(profile.index / 60, profile[col], color=color, label='Rata-rata hari kerja')
    ax.axhline(cap, color='red', ls='--', label=f'Kapasitas {cap}')
    ax.set_title(col); ax.set_xlabel('Jam'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()